## 1. NLTK movie_reviews 데이터로 RNN 모델 학습

In [1]:
import random
import zipfile
import requests
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer
import nltk
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [2]:
nltk.download('movie_reviews')
from nltk.corpus import movie_reviews

texts = []
labels = []
for fileid in movie_reviews.fileids():
    category = movie_reviews.categories(fileid)[0]
    texts.append(movie_reviews.raw(fileid))
    labels.append(1 if category == 'pos' else 0)

len(texts), sum(labels)

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\dohy3\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


(2000, 1000)

In [3]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN = 200

def preprocessing(text, max_length=MAX_LEN):
    if pd.isna(text):
        text = ''
    result = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='np'
    )
    return result['input_ids'][0]

In [4]:
mr_train_texts, mr_val_texts, mr_train_labels, mr_val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

len(mr_train_texts), len(mr_val_texts)

(1600, 400)

In [5]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, max_length=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = preprocessing(self.texts[idx], self.max_length)
        return {
            'tokens': torch.tensor(tokens, dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.float)
        }

mr_train_dataset = SentimentDataset(mr_train_texts, mr_train_labels)
mr_val_dataset = SentimentDataset(mr_val_texts, mr_val_labels)

mr_train_loader = DataLoader(mr_train_dataset, batch_size=16, shuffle=True)
mr_val_loader = DataLoader(mr_val_dataset, batch_size=16, shuffle=False)

In [6]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, tokens):
        embed = self.embedding(tokens)
        _, (hidden, _) = self.rnn(embed)
        hidden = hidden.squeeze(0)
        logit = self.classifier(hidden)
        return logit.squeeze(-1)

In [7]:
VOCAB_SIZE = tokenizer.vocab_size
EMBED_DIM = 128
HIDDEN_DIM = 64

model = SentimentRNN(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [8]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            tokens = batch['tokens'].to(device)
            label = batch['label'].to(device)
            logit = model(tokens)
            pred = (torch.sigmoid(logit) > 0.5).float()
            correct += (pred == label).sum().item()
            total += label.size(0)
    return correct / total

def train(model, train_loader, val_loader, criterion, optimizer, epochs=5):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch in train_loader:
            tokens = batch['tokens'].to(device)
            label = batch['label'].to(device)

            optimizer.zero_grad()
            logit = model(tokens)
            loss = criterion(logit, label)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        val_acc = evaluate(model, val_loader)
        print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}, Val Acc: {val_acc:.4f}')

In [9]:
train(model, mr_train_loader, mr_val_loader, criterion, optimizer, epochs=5)

Epoch 1, Loss: 0.6953, Val Acc: 0.4850


Epoch 2, Loss: 0.6703, Val Acc: 0.5075


Epoch 3, Loss: 0.5723, Val Acc: 0.4625


Epoch 4, Loss: 0.3543, Val Acc: 0.4600


Epoch 5, Loss: 0.1333, Val Acc: 0.4675


## 2. GLUE SST-2 데이터 다운로드

In [10]:
SST2_URL = 'https://dl.fbaipublicfiles.com/glue/data/SST-2.zip'
ZIP_PATH = 'SST-2.zip'
DATA_DIR = '.'

response = requests.get(SST2_URL, timeout=60)
with open(ZIP_PATH, 'wb') as f:
    f.write(response.content)

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(DATA_DIR)

import os
os.listdir('SST-2')

['dev.tsv', 'original', 'test.tsv', 'train.tsv']

## 3. SST-2 train.tsv로 Fine-tuning

In [11]:
sst2_train_df = pd.read_csv('SST-2/train.tsv', sep='\t')
sst2_train_df.head()

,sentence,label
0,hide new secretions from the parental units,0
1,"contains no wit , only labored gags",0
2,that loves its characters and communicates som...,1
3,remains utterly satisfied to remain the same t...,0
4,on the worst revenge-of-the-nerds clichés the ...,0


In [12]:
sst2_train_texts, sst2_val_texts, sst2_train_labels, sst2_val_labels = train_test_split(
    sst2_train_df['sentence'].tolist(),
    sst2_train_df['label'].tolist(),
    test_size=0.1, random_state=42, stratify=sst2_train_df['label'].tolist()
)

SST2_MAX_LEN = 64

sst2_train_dataset = SentimentDataset(sst2_train_texts, sst2_train_labels, max_length=SST2_MAX_LEN)
sst2_val_dataset = SentimentDataset(sst2_val_texts, sst2_val_labels, max_length=SST2_MAX_LEN)

sst2_train_loader = DataLoader(sst2_train_dataset, batch_size=32, shuffle=True)
sst2_val_loader = DataLoader(sst2_val_dataset, batch_size=32, shuffle=False)

In [13]:
finetune_optimizer = optim.Adam(model.parameters(), lr=5e-4)
train(model, sst2_train_loader, sst2_val_loader, criterion, finetune_optimizer, epochs=5)

Epoch 1, Loss: 0.6667, Val Acc: 0.7492


Epoch 2, Loss: 0.4103, Val Acc: 0.8624


Epoch 3, Loss: 0.2665, Val Acc: 0.8869


Epoch 4, Loss: 0.2001, Val Acc: 0.8968


Epoch 5, Loss: 0.1592, Val Acc: 0.9096


## 4. SST-2 test.tsv 추론 및 (index, label) 생성

In [14]:
sst2_test_df = pd.read_csv('SST-2/test.tsv', sep='\t')
sst2_test_df.head()

,index,sentence
0,0,uneasy mishmash of styles and genres .
1,1,this film 's relationship to actual tension is...
2,2,"by the end of no such thing the audience , lik..."
3,3,director rob marshall went out gunning to make...
4,4,lathan and diggs have considerable personal ch...


In [15]:
class SST2TestDataset(Dataset):
    def __init__(self, df, max_length=SST2_MAX_LEN):
        self.df = df.reset_index(drop=True)
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokens = preprocessing(row['sentence'], self.max_length)
        return {
            'index': int(row['index']),
            'tokens': torch.tensor(tokens, dtype=torch.long)
        }

sst2_test_dataset = SST2TestDataset(sst2_test_df)
sst2_test_loader = DataLoader(sst2_test_dataset, batch_size=64, shuffle=False)

In [16]:
model.eval()
indices = []
preds = []

with torch.no_grad():
    for batch in sst2_test_loader:
        tokens = batch['tokens'].to(device)
        logit = model(tokens)
        pred = (torch.sigmoid(logit) > 0.5).long().cpu().numpy()

        indices.extend(batch['index'].tolist())
        preds.extend(pred.tolist())

submission = pd.DataFrame({'index': indices, 'label': preds})
submission = submission.sort_values('index').reset_index(drop=True)
submission.to_csv('SST-2_test_predictions.tsv', sep='\t', index=False)
submission.head()

,index,label
0,0,0
1,1,0
2,2,1
3,3,1
4,4,1
